In [1]:
import pandas as pd
import glob
import os

RAW_DIR = '../../data/norming_study/raw'

In [2]:
# load all half csvs (skip demographics)
half_files = glob.glob(os.path.join(RAW_DIR, '*_half.csv'))
raw = pd.concat([pd.read_csv(f) for f in half_files], ignore_index=True)

print(f'{len(half_files)} files loaded, {len(raw)} total rows')
raw.head()

11 files loaded, 107 total rows


,trial_type,subjectID,prolificID,studyID,sessionID,DEBUG,sliderOrder,half,targetValue,passed,...,actionPhrase,abilityResponse,abilityRT,abilityDragged,willingnessResponse,willingnessRT,willingnessDragged,trialRT,trialIndex,suspicious
0,whyask-trial,tpku9734jv,NaN,NaN,NaN,0,AW,2,NaN,NaN,...,give me $50,95,10022.0,True,50,6864.0,True,16187.0,11,False
1,whyask-trial,tpku9734jv,NaN,NaN,NaN,0,AW,2,NaN,NaN,...,introduce two friends who would get along,96,3758.0,True,79,12193.0,True,22766.0,12,False
2,whyask-trial,tpku9734jv,NaN,NaN,NaN,0,AW,2,NaN,NaN,...,cook dinner for a small group of friends,75,3474.0,True,56,12794.0,True,35457.0,13,False
3,whyask-trial,tpku9734jv,NaN,NaN,NaN,0,AW,2,NaN,NaN,...,wake up early to exercise,89,5617.0,True,33,28740.0,True,36873.0,14,False
4,whyask-trial,tpku9734jv,NaN,NaN,NaN,0,AW,2,NaN,NaN,...,share your salary with a coworker,75,4674.0,True,7,18157.0,True,36607.0,15,False


In [3]:
# split trials vs attn checks
trials = raw[raw['half'].isin([1, 2])].copy()
attn   = raw[raw['half'] == 'attn'].copy()

print(f'trials: {len(trials)} rows, {trials["subjectID"].nunique()} subjects')
print(f'attn checks: {len(attn)} rows')
print(f'subjects: {sorted(trials["subjectID"].unique())}')

trials: 52 rows, 6 subjects
attn checks: 10 rows
subjects: ['369n4euwyd', '4dhnekfjvy', '6e7mdwq44t', 'jys1p0prv7', 'tpku9734jv', 'xj0mmuh0bf']


In [4]:
# load demographics
demo_files = glob.glob(os.path.join(RAW_DIR, '*_demographics.csv'))
demo = pd.concat([pd.read_csv(f) for f in demo_files], ignore_index=True)

print(f'{len(demo)} demo rows')
demo[['subjectID', 'age', 'gender', 'race', 'education', 'totalDurationMs']]

5 demo rows


,subjectID,age,gender,race,education,totalDurationMs
0,369n4euwyd,24.0,Female,White,bachelors,387632
1,jys1p0prv7,NaN,NaN,NaN,NaN,917739
2,xj0mmuh0bf,24.0,Female,Asian,bachelors,334154
3,6e7mdwq44t,27.0,Male,White,bachelors,570334
4,tpku9734jv,23.0,Female,Asian,bachelors,993025


In [5]:
# attn check summary — did they pass?
attn_cols = ['subjectID', 'itemID', 'abilityResponse', 'willingnessResponse', 'suspicious']
if 'targetValue' in attn.columns:
    attn_cols.insert(4, 'targetValue')
print(attn[attn_cols])



     subjectID  itemID  abilityResponse  willingnessResponse  targetValue  \
10  tpku9734jv  ATTN_1               79                   79         79.0   
11  tpku9734jv  ATTN_2               14                   14         14.0   
34  xj0mmuh0bf  ATTN_1               79                   79          NaN   
35  xj0mmuh0bf  ATTN_2               43                   43          NaN   
56  jys1p0prv7  ATTN_1               22                   22         22.0   
57  jys1p0prv7  ATTN_2               14                   14         14.0   
83  6e7mdwq44t  ATTN_1               43                   43         43.0   
84  6e7mdwq44t  ATTN_2               14                   14         14.0   
95  369n4euwyd  ATTN_1               86                   86         86.0   
96  369n4euwyd  ATTN_2               68                   68         68.0   

    suspicious  
10       False  
11       False  
34       False  
35       False  
56       False  
57       False  
83       False  
84       False  

In [6]:
# subjects who failed any attn check (suspicious == True on any row)
failed = attn[attn['suspicious'] == True]['subjectID'].unique()
print(f'failed attn: {failed}')

trials_clean = trials[~trials['subjectID'].isin(failed)].copy()
demo_clean   = demo[~demo['subjectID'].isin(failed)].copy()

print(f'kept {trials_clean["subjectID"].nunique()} subjects ({len(trials_clean)} trials)')

failed attn: []
kept 6 subjects (52 trials)


In [7]:
# quick look at trial responses
print(trials[['subjectID', 'itemID', 'actionPhrase', 'abilityResponse', 'willingnessResponse', 'sliderOrder']].to_string())
print()
# avg time per trial
avg_rt = trials['trialRT'].mean() / 1000
print(f'avg trial RT: {avg_rt:.1f}s')

      subjectID itemID                                      actionPhrase  abilityResponse  willingnessResponse sliderOrder
12   4dhnekfjvy     98           hold something for someone for a moment               65                   61          AW
13   4dhnekfjvy     78       help someone pick up something they dropped               73                   59          AW
14   4dhnekfjvy     37            let a classmate copy your exam answers               78                   59          AW
15   4dhnekfjvy     76                scoot over on a bench to make room               79                   65          AW
16   4dhnekfjvy     55  lend your car to an acquaintance for the weekend               81                   64          AW
17   4dhnekfjvy     14                 sing a very high note in an opera               87                   66          AW
18   4dhnekfjvy     69                      tell someone what time it is               77                   61          AW
19   6e7mdwq44t 

In [8]:
# save processed
PROC_DIR = '../../data/norming_study/processed'
os.makedirs(PROC_DIR, exist_ok=True)

# drop trial-level cols from demo
trial_cols = ['trialIndex', 'itemID', 'actionPhrase', 'abilityResponse', 'abilityRT',
              'abilityDragged', 'willingnessResponse', 'willingnessRT', 'willingnessDragged',
              'trialRT', 'suspicious', 'targetValue', 'passed']
demo_out = demo_clean.drop(columns=[c for c in trial_cols if c in demo_clean.columns])

trials_clean.to_csv(os.path.join(PROC_DIR, 'trials.csv'), index=False)
demo_out.to_csv(os.path.join(PROC_DIR, 'demographics.csv'), index=False)

print(f'saved trials.csv ({len(trials_clean)} rows) and demographics.csv ({len(demo_out)} rows)')
print(f'demo cols: {list(demo_out.columns)}')

saved trials.csv (52 rows) and demographics.csv (5 rows)
demo cols: ['trial_type', 'subjectID', 'prolificID', 'studyID', 'sessionID', 'DEBUG', 'sliderOrder', 'half', 'age', 'gender', 'race', 'education', 'strategy', 'technicalIssues', 'feedback', 'visibilityChanges', 'totalDurationMs']
